# QA Generator v2 — few-shot, seeded with the hand-written items

Scales the Wikipedia-grounded benchmark past the 200 hand-written items by showing the model real examples from that set, so generated items inherit its question style, Darija register and answer format.

Includes an **inline auto-reject** stage: items with no Darija markers, invented numbers, or low passage overlap are discarded and retried before they ever reach the validation notebook.

**Upload:** `corpus_v2.json`, `qa_pairs_wiki.json`. **Secret:** `GROQ_API_KEY`.

### Install

In [ ]:
# !pip install -q groq tqdm

### Config

In [ ]:
CONFIG = {
    "n_target": 300,              # how many NEW items to generate (on top of the 200 hand-written)
    "n_fewshot": 4,               # examples shown per prompt (rotated, so output isn't formulaic)
    "model": "openai/gpt-oss-20b",
    "max_tokens": 900,
    "temperature": 0.4,           # some variety; too low makes every question identical in shape
    "sleep": 0.3,
    "checkpoint_every": 25,
    "id_prefix": "g",             # generated items get g001... (hand-written are w001-w200)
    "min_passage_chars": 450,
    "max_passage_chars": 900,
    "passage_snippet_chars": 450, # how much of the example passage to show in the prompt
    "max_attempts_per_passage": 2,
}

### Load corpus, existing items, and pick few-shot seeds

In [ ]:
import json, random, re

with open("corpus_v2.json", encoding="utf-8") as f:
    corpus = json.load(f)
with open("qa_pairs_wiki.json", encoding="utf-8") as f:
    hand_written = json.load(f)

corpus_map = {c["chunk_id"]: c["text"] for c in corpus}
used_chunks = {q["source_chunk_id"] for q in hand_written}

def is_junk(t):
    """Same filter used when selecting passages for the hand-written batches:
    drops reference lists, bibliographies and name dumps that can't support a
    good question."""
    if len(re.findall(r"[A-Za-z]", t)) / max(len(t), 1) > 0.08:
        return True
    for m in ["DOI:", "ISBN", "ISSN", "مؤرشف من الأصل", "اطلع عليه بتاريخ", "واي باك مشين"]:
        if m in t:
            return True
    if len(re.findall(r"\d{4}\s*[-–]\s*\d{4}", t)) >= 4:
        return True
    if t.count(":") >= 8:
        return True
    return sum(t.count(w) for w in ["في ", "من ", "التي ", "الذي ", "حيث ", "كما ", "على ", "إلى "]) < 8

candidates = [
    c for c in corpus
    if c.get("source") == "wikipedia_ar"
    and CONFIG["min_passage_chars"] <= len(c["text"]) <= CONFIG["max_passage_chars"]
    and c["chunk_id"] not in used_chunks
    and not is_junk(c["text"])
]

random.seed(2024)
random.shuffle(candidates)
sampled = candidates[: CONFIG["n_target"]]

# Few-shot pool: hand-written items whose passage is available, spanning
# different question types (how many / how far / why / what / when).
FEWSHOT_IDS = ["w043", "w079", "w193", "w182", "w172", "w003", "w116", "w125"]
fewshot_pool = []
for wid in FEWSHOT_IDS:
    match = [x for x in hand_written if x["id"] == wid]
    if match and match[0]["source_chunk_id"] in corpus_map:
        q = match[0]
        fewshot_pool.append({
            "passage": corpus_map[q["source_chunk_id"]][: CONFIG["passage_snippet_chars"]],
            "msa_query": q["msa_query"],
            "darija_query": q["darija_query"],
            "gold_answer": q["gold_answer"],
        })

print(f"Corpus: {len(corpus)} | hand-written: {len(hand_written)} | usable new passages: {len(candidates)}")
print(f"Will generate from {len(sampled)} passages using {len(fewshot_pool)} few-shot seeds.")

### Groq client

In [ ]:
import time
from groq import Groq, RateLimitError
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY not found in Colab Secrets (key icon, left sidebar).")
client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready.")

### Prompt builder (few-shot) and response parser

In [ ]:
INSTRUCTIONS = """أنت مساعد متخصص في إنشاء بيانات لتقييم أنظمة الاسترجاع والبحث باللغة العربية.

لكل فقرة، أنشئ:
1. سؤالاً بالعربية الفصحى، إجابته مذكورة صراحةً في الفقرة.
2. نفس السؤال بالدارجة المغربية كما يطرحه مغربي في حديث عادي.
3. إجابة قصيرة ودقيقة مأخوذة حرفياً من الفقرة.

قواعد صارمة:
- لا تخترع أي معلومة؛ الإجابة يجب أن تكون موجودة في الفقرة.
- انسخ الأرقام والتواريخ والأسماء كما هي مكتوبة في الفقرة بالضبط.
- صيغة الدارجة يجب أن تختلف فعلياً عن الفصحى، وتستعمل كلمات دارجة حقيقية
  (شنو، شحال، فين، علاش، كيفاش، واش، ديال، شكون، إمتى، أشمن، بشحال، كاين).
- اجعل السؤال محدداً بحيث لا يمكن الإجابة عليه إلا من هذه الفقرة بالذات.
- تجنب الأسئلة العامة التي تصلح لأي فقرة.

أجب بصيغة JSON فقط:
{"msa_query": "...", "darija_query": "...", "gold_answer": "..."}"""


def build_prompt(passage_text, shots):
    parts = [INSTRUCTIONS, "\n\nأمثلة:\n"]
    for s in shots:
        example = {
            "msa_query": s["msa_query"],
            "darija_query": s["darija_query"],
            "gold_answer": s["gold_answer"],
        }
        parts.append(f'\nالفقرة:\n"""{s["passage"]}"""\n')
        parts.append(json.dumps(example, ensure_ascii=False))
        parts.append("\n")
    parts.append(f'\nالآن الفقرة الجديدة:\n"""{passage_text}"""\n')
    return "".join(parts)


def parse_json(text):
    text = (text or "").replace("```json", "").replace("```", "").strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        obj = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    if not all(k in obj and str(obj[k]).strip() for k in ("msa_query", "darija_query", "gold_answer")):
        return None
    return {k: str(obj[k]).strip() for k in ("msa_query", "darija_query", "gold_answer")}


def generate(passage_text, max_retries=4):
    shots = random.sample(fewshot_pool, min(CONFIG["n_fewshot"], len(fewshot_pool)))
    prompt = build_prompt(passage_text, shots)
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=CONFIG["model"],
                messages=[{"role": "user", "content": prompt}],
                temperature=CONFIG["temperature"],
                max_tokens=CONFIG["max_tokens"],
                reasoning_effort="low",
            )
            parsed = parse_json(r.choices[0].message.content)
            if parsed:
                return parsed
        except RateLimitError:
            wait = 20 * (attempt + 1)
            print(f"  Rate limited, waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"  Generation error: {e}")
            return None
    return None

### Inline auto-reject (cheap checks applied as items are produced)

In [ ]:
DARIJA_MARKERS = [
    "ديال", "بزاف", "واخا", "دابا", "غادي", "ماشي", "شنو", "علاش", "فين", "كيفاش",
    "بغيت", "كاين", "ماكاينش", "منين", "واش", "بشحال", "شحال", "إمتى", "شكون",
    "فأش", "فأي", "أشمن", "أش", "ملي", "حيت", "راه", "بحال", "هادشي", "هاد", "كي", "كت",
]

def auto_reject(item, passage_text):
    """Returns a rejection reason, or None if the item passes.
    Mirrors the cheap checks from the validation notebook so obviously broken
    items are discarded before they cost review time."""
    msa, dar, ans = item["msa_query"], item["darija_query"], item["gold_answer"]

    if msa.strip() == dar.strip():
        return "darija identical to msa"
    if not any(m in dar for m in DARIJA_MARKERS):
        return "no darija markers"
    if len(msa) < 10 or len(dar) < 10 or len(ans) < 2:
        return "too short"

    # Every number in the answer must appear verbatim in the passage.
    nums = re.findall(r"\d[\d٬.,]*", ans)
    for n in nums:
        if n.rstrip(".,") and n.rstrip(".,") not in passage_text:
            return f"number {n} not in passage"

    # Answer should overlap the passage lexically; a fully novel answer is a
    # strong signal the model invented it rather than extracting it.
    ans_words = [w for w in re.findall(r"\w+", ans) if len(w) > 3]
    if ans_words:
        overlap = sum(1 for w in ans_words if w in passage_text) / len(ans_words)
        if overlap < 0.5:
            return f"low passage overlap ({overlap:.0%})"
    return None

### Generate with checkpointing

In [ ]:
import os
from tqdm.auto import tqdm

CHECKPOINT = "gen_checkpoint.json"
generated, rejected, start = [], [], 0

if os.path.exists(CHECKPOINT):
    state = json.load(open(CHECKPOINT, encoding="utf-8"))
    generated, rejected, start = state["generated"], state["rejected"], state["next_index"]
    print(f"Resuming: {len(generated)} kept, {len(rejected)} rejected, next passage #{start}")

for i in tqdm(range(start, len(sampled)), initial=start, total=len(sampled)):
    passage = sampled[i]
    kept = False

    for _ in range(CONFIG["max_attempts_per_passage"]):
        item = generate(passage["text"])
        if not item:
            break
        reason = auto_reject(item, passage["text"])
        if reason is None:
            generated.append({
                "id": f"{CONFIG['id_prefix']}{len(generated)+1:03d}",
                "msa_query": item["msa_query"],
                "darija_query": item["darija_query"],
                "gold_answer": item["gold_answer"],
                "source_chunk_id": passage["chunk_id"],
            })
            kept = True
            break
        time.sleep(CONFIG["sleep"])

    if not kept:
        rejected.append({"chunk_id": passage["chunk_id"], "reason": reason if item else "generation failed"})

    if (i + 1) % CONFIG["checkpoint_every"] == 0:
        json.dump({"generated": generated, "rejected": rejected, "next_index": i + 1},
                  open(CHECKPOINT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    time.sleep(CONFIG["sleep"])

json.dump({"generated": generated, "rejected": rejected, "next_index": len(sampled)},
          open(CHECKPOINT, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"\nKept: {len(generated)} | Rejected: {len(rejected)}")

### Rejection breakdown (tells you if the prompt needs work)

In [ ]:
from collections import Counter

reasons = Counter(r["reason"] for r in rejected)
print("Rejection reasons:")
for reason, n in reasons.most_common():
    print(f"  {n:4d}  {reason}")

keep_rate = len(generated) / max(len(sampled), 1)
print(f"\nKeep rate: {keep_rate:.1%}")
if keep_rate < 0.6:
    print("Keep rate is low — worth inspecting samples below before generating more.")

### Eyeball a sample before trusting the batch

In [ ]:
for item in random.sample(generated, min(8, len(generated))):
    print("-" * 70)
    print("MSA:    ", item["msa_query"])
    print("Darija: ", item["darija_query"])
    print("Answer: ", item["gold_answer"])
    print("Passage:", corpus_map[item["source_chunk_id"]][:200], "...")

### Merge with hand-written items and save

In [ ]:
combined = hand_written + generated

with open("qa_pairs_generated.json", "w", encoding="utf-8") as f:
    json.dump(generated, f, ensure_ascii=False, indent=2)
with open("qa_pairs_wiki_v2.json", "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print(f"Hand-written: {len(hand_written)}")
print(f"Generated:    {len(generated)}")
print(f"Wikipedia benchmark total: {len(combined)}")

from google.colab import files
files.download("qa_pairs_generated.json")
files.download("qa_pairs_wiki_v2.json")

# Next: run dataset_validation_v2.ipynb on qa_pairs_wiki_v2.json.
# Generated items will flag at a higher rate than hand-written ones — that gap
# is itself worth reporting in the paper's dataset section.